# PathSnap: exploratory data analysis (Phase 1)

Run from the repo root. Tables: `data/processed/*.csv` (partitions: **train 80 trips + student_test 10 trips**). Raw-file plots use `Student folder/train` (**train only**). The evaluator partition is never used.

The `# YOUR OBSERVATION` cells are for your own words: the report must reflect what you understand (see the academic-integrity rules).

In [ ]:
import sys; sys.path.insert(0, '../..')
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
from src.viz import eda

PROC, RAW, FIG = Path('../../data/processed'), Path('../../Student folder/train'), Path('../../results/figures')
inv, net, iss, both = eda.load_tables(PROC)
pd.read_csv(PROC / 'candidate_summary.csv')

## 1. Trips per partition and points per trip

In [ ]:
display(inv.partition.value_counts().to_frame('trips'))
display(inv.groupby('partition').n_obs.describe().round(0))
display(Image(eda.fig_inventory(inv, FIG)))

**YOUR OBSERVATION:** (spread of trip lengths, any very short or very long trips)

## 2. Time gaps between GPS points

In [ ]:
steps = eda.raw_dt_speed(RAW)   # train raw files only
display(Image(eda.fig_time_gaps(inv, steps, FIG)))
inv.sort_values('dt_max_s', ascending=False)[['trip_id','partition','n_obs','dt_max_s','n_gaps_gt_5s']].head(6)

Trips `00000090` and `00000078` have one huge timestamp jump each (153,064 s and 52,810 s, at the end and start of the trip), so `duration_s` and mean speed are not meaningful for them.

**YOUR OBSERVATION:**

## 3. Speed outliers

In [ ]:
display(Image(eda.fig_speed(inv, steps, FIG)))
inv[inv.n_implausible_speed > 0][['trip_id','partition','max_speed_kmh','n_implausible_speed']]

Speed spikes concentrate in two train trips: `00000013` (95 steps above 250 km/h) and `00000032` (40 steps). Nine other trips have 1-3 such steps (single GPS jumps, e.g. `00000079` reaches 649 km/h).

**YOUR OBSERVATION:**

## 4. Road-network size and connectivity

In [ ]:
display(Image(eda.fig_network(both, FIG)))
both[['n_nodes','n_arcs','n_weak_components','largest_weak_component_frac','n_isolated_nodes','two_way_arc_frac']].describe().round(3)

**YOUR OBSERVATION:** (no isolated nodes, but many weak components: what does that imply for the HMM transition model?)

## 5. Data-quality issues and reference-route quality

In [ ]:
display(iss.groupby('issue').agg(trips=('trip_id','nunique'), total=('count','sum')))
display(Image(eda.fig_issues(iss, FIG)))
display(Image(eda.fig_label_quality(inv, FIG)))

## 6. Geographic extent

In [ ]:
display(Image(eda.fig_extent(inv, FIG)))

## 7. Case studies: trips 33 and 54 (labels unreliable) and a normal trip (10)

Static views here; the interactive Folium versions come from `python scripts/make_eda_maps.py` (`results/maps/*.html`).

In [ ]:
for t in ['00000033', '00000054', '00000010']:
    display(Image(eda.fig_trip_overview(RAW / t, FIG)))

**YOUR OBSERVATION:** (see `report/eda_findings.md` for the numbers behind 33 and 54)